In [5]:
!pip install pypdf pandas

In [10]:
!pip install datasets google-generativeai

INFO: pip is looking at multiple versions of googleapis-common-protos to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of proto-plus to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.5 MB/s  0:00:00
  Attempting uninstall: protobuf
    Found existing installation: prot

In [9]:
from datasets import load_dataset
import random

# Load the expert-labeled subset
ds = load_dataset("qiaojin/PubMedQA", "pqa_labeled")

# The labeled dataset is loaded into 'train' by default.
# Shuffle and select 50 fixed instances for reproducible benchmarking.
experiment_sample = ds['train'].shuffle(seed=42).select(range(50))

# Example of extracting the variables for your RAG and Debate prompts
for item in experiment_sample:
    question = item['question']

    # PubMedQA stores context as a list of strings; join them for your static RAG prompt
    rag_context = "\n".join(item['context'])
    reference_rationale = item['long_answer']

    # Route `question` to your 2-Agent Debate loop
    # Route `question` + `rag_context` to your Single LLM RAG baseline
    break

In [12]:
from datasets import load_dataset

# Load the expert-labeled dataset
ds = load_dataset("qiaojin/PubMedQA", "pqa_labeled")

# Extract the first row
sample_row = ds['train'][0]

print("--- VERIFYING DATASET KEYS ---")
print(f"Available Keys: {list(sample_row.keys())}\n")

print("--- VERIFYING 'question' ---")
print(sample_row['question'], "\n")

print("--- VERIFYING 'contexts' ---")
print(type(sample_row['context'])) # Should be a list
print(sample_row['context'])

--- VERIFYING DATASET KEYS ---
Available Keys: ['pubid', 'question', 'context', 'long_answer', 'final_decision']

--- VERIFYING 'question' ---
Do mitochondria play a role in remodelling lace plant leaves during programmed cell death? 

--- VERIFYING 'contexts' ---
<class 'dict'>
{'contexts': ['Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants.', 'The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a win

In [18]:
import google.generativeai as genai
from google.colab import userdata

genai.configure(api_key=userdata.get('secretName'))

print("All available models for text generation:")
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

All available models for text generation:
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/lyria-3-clip-p

In [24]:
import google.generativeai as genai
from google.colab import userdata

# Configure Gemini API (Assuming you have stored your key in Colab Secrets as 'GEMINI_API_KEY')
genai.configure(api_key=userdata.get('secretName'))
model = genai.GenerativeModel('gemini-3.8-flash')

def generate_and_log_tokens(prompt_text, pipeline_name):
    """Generates content and extracts token counts."""
    response = model.generate_content(prompt_text)

    # Extract token usage from the response metadata
    in_tokens = response.usage_metadata.prompt_token_count
    out_tokens = response.usage_metadata.candidates_token_count

    print(f"[{pipeline_name}] Input Tokens: {in_tokens} | Output Tokens: {out_tokens} | Total: {in_tokens + out_tokens}")
    return response.text, in_tokens, out_tokens

# Prepare inputs from the sample row
question = sample_row['question']
rag_context = "\n".join(sample_row['context'])

print("--- VERIFYING TOKEN LOGS ---")

# Pipeline 1: Control (Zero-Shot)
control_prompt = f"Answer this scientific question: {question}"
_, in_ctrl, out_ctrl = generate_and_log_tokens(control_prompt, "Control Baseline")

# Pipeline 2: RAG (Context + Question)
rag_prompt = f"Context:\n{rag_context}\n\nBased on the context, answer this scientific question: {question}"
_, in_rag, out_rag = generate_and_log_tokens(rag_prompt, "RAG Baseline")

# Pipeline 3: Debate (2-Round Loop Example)
# Round 1: Agent 1 proposes an answer
agent1_prompt = f"Propose a scientific rationale to answer this question: {question}"
draft_rationale, in_d1, out_d1 = generate_and_log_tokens(agent1_prompt, "Debate - Round 1")

# Round 2: Agent 2 critiques and refines
agent2_prompt = f"Critique and refine the following scientific rationale for accuracy.\nRationale: {draft_rationale}"
_, in_d2, out_d2 = generate_and_log_tokens(agent2_prompt, "Debate - Round 2")

print(f"[Debate - Cumulative] Total Input: {in_d1 + in_d2} | Total Output: {out_d1 + out_d2}")

--- VERIFYING TOKEN LOGS ---
[Control Baseline] Input Tokens: 22 | Output Tokens: 467 | Total: 489
[RAG Baseline] Input Tokens: 50 | Output Tokens: 217 | Total: 267
[Debate - Round 1] Input Tokens: 27 | Output Tokens: 1018 | Total: 1045
[Debate - Round 2] Input Tokens: 1033 | Output Tokens: 1837 | Total: 2870
[Debate - Cumulative] Total Input: 1060 | Total Output: 2855


In [26]:
import time

# Ensure you are using the same model that worked for you in Checklist 2
judge_model = genai.GenerativeModel(
    'gemini-3.8-flash',
    system_instruction="You are a strict scientific judge. You evaluate hypotheses on a scale of 1 to 10. You must ONLY output a single integer. No words, no punctuation, no markdown."
)

judge_prompt_template = """
Evaluate the plausibility of the following scientific rationale on a scale of 1 to 10.
Question: {question}
Rationale: {rationale}
"""

print("--- VERIFYING JUDGE PARSING ---")

for i in range(1, 6):
    prompt = judge_prompt_template.format(question=question, rationale=draft_rationale)

    try:
        response = judge_model.generate_content(prompt)
        raw_output = response.text.strip()

        # Attempt to cast the LLM's text output directly to a Python integer
        score = int(raw_output)
        print(f"Trial {i}: SUCCESS - Parsed score -> {score}")

    except ValueError:
        print(f"Trial {i}: FAILED - Could not parse integer. Raw output: '{raw_output}'")
    except Exception as e:
        print(f"Trial {i}: FAILED - API Error: {e}")

    # Pause for 5 seconds before the next API call to prevent 504 Timeouts
    print("Waiting 5 seconds...")
    time.sleep(5)

--- VERIFYING JUDGE PARSING ---
Trial 1: SUCCESS - Parsed score -> 10
Waiting 5 seconds...
Trial 2: SUCCESS - Parsed score -> 10
Waiting 5 seconds...
Trial 3: SUCCESS - Parsed score -> 10
Waiting 5 seconds...
Trial 4: SUCCESS - Parsed score -> 10
Waiting 5 seconds...
Trial 5: FAILED - API Error: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.
Please retry in 14h39m47.549735601s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  description: "G